# 第 1 章 · LangChain 生态版图与模型接入

> 本章目标：
> 1. 看清 LangChain "家族"的全貌——它不是**一个**库，而是一套生态；
> 2. 理解 LangChain 1.x 时代的开发范式转变；
> 3. 用 DeepSeek 跑通模型调用、流式输出与批量处理。

---

## 1. LangChain 生态版图

刚接触 LangChain 的人最容易犯的错，是把"LangChain"当成一个单一框架。实际上它是一个**分层生态**：

```mermaid
flowchart TB
    subgraph CORE["地基"]
        LCORE["langchain-core<br/>消息/Runnable/工具 抽象接口"]
    end
    subgraph MID["集成层"]
        LC["langchain<br/>Agent 构建（create_agent）"]
        LCOM["langchain-community<br/>社区集成（几百个）"]
        PARTNER["partner 包<br/>langchain-openai / -anthropic / ..."]
    end
    subgraph TOP["编排与平台"]
        LG["LangGraph<br/>状态图编排引擎"]
        LS["LangSmith<br/>观测/评估/数据集 SaaS"]
        LP["LangGraph Platform<br/>托管部署"]
    end
    CORE --> MID --> TOP
    LC --> LG
    style LCORE fill:#e6f4ea,stroke:#34a853,stroke-width:2px
    style LG fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style LS fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

| 包 | 职责 | 你需要它吗？ |
|---|---|---|
| `langchain-core` | 定义消息、Runnable、工具等**抽象** | 永远需要（被传递依赖） |
| `langchain` | **Agent 构建入口**（1.x 起聚焦 `create_agent`） | 需要 |
| `langchain-openai` 等 partner 包 | 各家模型/服务的**官方集成** | 按需安装 |
| `langchain-community` | 社区贡献的集成（质量参差） | 按需 |
| `langgraph` | 状态图编排（第 4-6 章主角） | 构建复杂 Agent 需要 |
| `langsmith` | Trace、评估、数据集（云服务） | 生产强烈推荐 |

### LangChain 1.x：一次重要的"瘦身"

2025 年 10 月发布的 **LangChain 1.0** 是框架的成年礼，关键变化：

| 旧时代（0.x） | 1.x 时代 |
|---|---|
| 十几种 Agent 执行器（AgentExecutor、各种 REACT 变体） | **统一为 `create_agent`**（底层是 LangGraph） |
| 链（Chain）概念泛滥 | **LCEL** 一统天下，旧链移入 `langchain-classic` |
| 记忆方案五花八门 | 交给 LangGraph **Checkpointer** |
| 与 LangGraph 关系模糊 | 明确分工：**LangChain 管"造 Agent"，LangGraph 管"编排"** |

> 📌 本教程完全基于 1.x 范式编写。网上大量 0.x 时代的教程（`LLMChain`、`initialize_agent` 等）已过时，请注意甄别。

---

## 2. 接入 DeepSeek：OpenAI 兼容协议

DeepSeek 官方 API 遵循 **OpenAI 协议**，因此直接用 `langchain-openai` 的 `ChatOpenAI`，改两个参数即可：

| 参数 | 值 |
|---|---|
| `model` | `deepseek-chat`（V3，通用）/ `deepseek-reasoner`（R1，深度推理） |
| `base_url` | `https://api.deepseek.com` |
| `api_key` | 从环境变量 `DEEPSEEK_API_KEY` 读取 |


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

import importlib.metadata as im
for pkg in ["langchain", "langchain-core", "langchain-openai", "langgraph"]:
    print(f"{pkg:20s} {im.version(pkg)}")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
    temperature=0,
    max_tokens=1024,
)
print("\n✅ 模型客户端就绪")


langchain            1.2.15
langchain-core       1.6.0
langchain-openai     1.6.0
langgraph            1.2.11



✅ 模型客户端就绪


---

## 3. 消息模型：一切对话的基本单位

LangChain 中，与模型的所有交互都围绕**消息（Message）对象**展开，而非裸字符串：

| 消息类型 | 角色 | 类比 |
|---|---|---|
| `SystemMessage` | 系统指令 | 岗位说明书（≈ ADK 的 instruction） |
| `HumanMessage` | 用户输入 | 用户说的话 |
| `AIMessage` | 模型回复 | 助手的回答（含 tool_calls） |
| `ToolMessage` | 工具结果 | 工具执行完交回的报告 |

```mermaid
flowchart LR
    S["SystemMessage<br/>你是翻译官"] --> L["💬 messages 列表"]
    H["HumanMessage<br/>翻译：你好"] --> L
    L --> M["ChatModel.invoke()"]
    M --> A["AIMessage<br/>Hello"]
    style L fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```


In [2]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="你是严谨的数学老师，解题时先给思路再给答案。"),
    HumanMessage(content="一个水池，进水管 3 小时注满，排水管 6 小时排空。两管同开，几小时注满？"),
]

response = llm.invoke(messages)
print(type(response).__name__)   # AIMessage
print(response.content)
print("─" * 50)
print("token 用量：", response.usage_metadata)


AIMessage
好的，我们先来分析一下这个问题。  

**思路分析**：  
1. 进水管单独工作，3小时注满水池，说明它的注水速度是每小时注满水池的 \( \frac{1}{3} \)。  
2. 排水管单独工作，6小时排空水池，说明它的排水速度是每小时排掉水池的 \( \frac{1}{6} \)。  
3. 两管同时打开时，水池的净变化速度 = 注水速度 - 排水速度，即：  
\[
\frac{1}{3} - \frac{1}{6} = \frac{2}{6} - \frac{1}{6} = \frac{1}{6}
\]  
也就是说，每小时水池净增加 \( \frac{1}{6} \) 的水量。  
4. 要注满整个水池（即达到1个单位），所需时间为：  
\[
1 \div \frac{1}{6} = 6 \text{小时}
\]

**答案**：两管同开，需要 **6小时** 才能注满水池。  

这样，我们通过速度差的方式，把问题转化成了简单的除法计算。
──────────────────────────────────────────────────
token 用量： {'input_tokens': 50, 'output_tokens': 250, 'total_tokens': 300, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


> 💡 注意 `AIMessage` 不只是文本——它携带 `usage_metadata`（token 统计）、`tool_calls`（工具调用请求，第 3 章）、`response_metadata` 等元数据。**结构化消息**是 LangChain 一切上层能力的基石。

---

## 4. Runnable 接口：统一的调用约定

LangChain 生态中几乎所有组件（模型、提示词、解析器、检索器……）都实现同一个 **Runnable** 接口：

| 方法 | 作用 | 异步版 |
|---|---|---|
| `invoke(input)` | 单次调用 | `ainvoke` |
| `stream(input)` | 流式输出 | `astream` |
| `batch(inputs)` | 批量并发 | `abatch` |

> 📌 **这是 LangChain 最重要的设计**：统一的接口让组件可以像乐高一样拼接（第 2 章的 LCEL 就是建立在此之上）。

### 4.1 流式输出


In [3]:
print("流式输出：", end="", flush=True)
for chunk in llm.stream([HumanMessage(content="用三行小诗赞美杭州的秋天")]):
    print(chunk.content, end="", flush=True)
print()


流式输出：

##

 《

杭州

的

...

开

宋

词的

涟漪

。

### 4.2 批量并发

`batch` 会并发发起多个请求，适合"对一批输入做同样处理"的场景：


In [4]:
cities = ["北京", "上海", "成都"]
prompts = [[HumanMessage(content=f"用一个四字词语形容{c}的气质，只输出词语")] for c in cities]

results = llm.batch(prompts)
for city, r in zip(cities, results):
    print(f"{city} → {r.content}")


北京 → 大气磅礴
上海 → 海纳百川
成都 → 巴适安逸


---

## 5. 切换模型：生态的威力

`ChatOpenAI` 只是几十种 **ChatModel** 实现之一。换成其他厂商通常只需换一个类：

```python
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-sonnet-5")

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1")   # 本地模型
```

上层代码（invoke/stream/batch/LCEL/create_agent）**一行不用改**——这就是抽象接口的价值。与 ADK 的 `BaseLlm` + LiteLLM 思路完全同构，只是 LangChain 用"一个厂商一个包"的方式实现，类型适配更原生。

---

## 6. 与 ADK 对照 🔄

| 概念 | LangChain | ADK |
|---|---|---|
| 模型抽象 | `BaseChatModel`（ChatOpenAI 等） | `BaseLlm`（Gemini / LiteLlm） |
| 第三方模型接入 | partner 包各自实现 | 统一走 LiteLLM |
| 系统指令 | `SystemMessage`（每次调用显式传） | `Agent(instruction=...)`（声明一次） |
| 单次调用 | `llm.invoke(messages)` → AIMessage | `runner.run_async()` → Event 流 |
| 流式 | `llm.stream()` | `run_async()` 本身就是流式 |
| 会话状态 | 无（基础层无状态） | Session/State 内建 |

> 🔍 **关键差异**：LangChain 基础层是**无状态**的函数式组件（历史自己管理）；ADK 从第一天起就内建了会话服务。这个差异将在第 4 章 LangGraph 登场时被"补平"——Checkpointer 就是 LangChain 体系对 Session 的回答。

---

## 📌 本章要点回顾

- LangChain 是**分层生态**：core（抽象）→ 集成包 → LangGraph（编排）→ LangSmith（观测）；
- 1.x 范式：`create_agent` + LCEL + LangGraph，旧式 Chain/AgentExecutor 已成历史；
- 一切交互皆**消息对象**，一切组件皆 **Runnable**（invoke/stream/batch）；
- DeepSeek 走 OpenAI 兼容协议，`ChatOpenAI` 改 `base_url` 即用。

> ➡️ 下一章：[02-LCEL与链式编排](02-LCEL与链式编排.ipynb) —— 学习 LangChain 独有的"管道美学"。
